# CrisisQ — Selective-precision mitigation experiment v1.1

**Compatibility fix:** forces the Agg plotting backend before import, including dependency probes. Use Run All in a fresh session; no experiment settings changed.

**Same whole original model-containing Kaggle input; no retraining or classification-threshold change.** Internet ON; GPU not required. Attach the original output of `CrisisQ_nonclinical_claim_verification` under `/kaggle/input`, containing the primary `models/saqs__42/fp32.onnx` and `q_quint8.onnx`. No input ZIP or manual dataset upload.

**Research question:** can a fixed, label-free fallback rule recover compressed-model positive losses more effectively than random fallback at the same call rate, while retaining measured speed and reporting memory/alert costs?

The primary policy retains the frozen linguistic gate, runs compressed inference for unresolved messages, and calls the frozen FP32 model only for compressed-negative scores close below the ORIGINAL classification threshold. Existing compressed positives are retained. This property is by construction; it is not clinical safety. Additional alerts, latency and co-resident model memory are measured.

The new fallback width is calibrated without labels on the previously used256-message Twitter runtime panel, with a primary5% allowance among neural-routed calibration messages. It is NOT the classification threshold. Evaluation excludes those256rows and adds the pinned SDCNL authors' test source. The width is transferred unchanged to both runtimes and SDCNL. Actual shifted-data fallback rates may exceed5% and are reported.

**Primary policy: `one_sided_05`.** Other budgets and symmetric routing are fixed controls/sensitivity checks, not a policy search. Compare always compressed, always FP32, and same-count random negative fallback. Multi-runtime decisions and actual fresh-process Python pipeline timing are recorded; this is not a phone benchmark.

These sources were inspected in earlier work; this is a new intervention study on known corpora, not an untouched independent validation. Precision cascades have prior art, e.g. [CascadeCNN (2018)](https://arxiv.org/abs/1807.05053). Favorable results could strengthen CrisisQ's mitigation evidence, not establish a new general cascade algorithm or guarantee publication.


In [ ]:
from pathlib import Path
import base64,hashlib,importlib.util,io,json,os,subprocess,sys,zipfile
RUN_ENV=os.environ.copy()
RUN_ENV["MPLBACKEND"]="Agg"
RUN_ENV.pop("PYTHONPATH",None)
RUN_ENV.pop("PYTHONHOME",None)
INPUT=Path('/kaggle/input')
OUTPUT=Path('/kaggle/working/crisisq_selective_precision_v1_1')
TEMP=Path('/kaggle/temp/crisisq_selective_precision_v1_1')
BUNDLE=Path('/kaggle/working/_crisisq_selective_precision_v1_1_bundle')
# Fixed optional extension list: SAQS42/123/777 + CE42, only complete pairs.
# Primary is mandatory; missing extensions are documented, never substituted.
MODEL_SCOPE='fixed_extensions'  # 'primary_only' gives a narrower, faster study
CONFIG=dict(input_root=str(INPUT),output_root=str(OUTPUT),temporary_root=str(TEMP),
            model_scope=MODEL_SCOPE,runtime_versions=['1.31.1','1.28.0'],timing_rounds=3)
for p in [OUTPUT,TEMP,BUNDLE]:p.mkdir(parents=True,exist_ok=True)


In [ ]:
# Self-contained, hash-verified helper bundle. No model weights or raw posts.
PAYLOAD='...'
EXPECTED='f74b9c151d85f1c6108c448a8a6d7895bf8382f910d4760c1e8ab7da4f2da78d'
FILES={'HOW_TO_RUN.txt': 'd08f42c99ab95f3649d4fa1b5c7a98d8ac3f5186b8e99e4c99befce27270c9fb', 'PROTOCOL.txt': 'a31243d3aa622faa1453b8e4980114fd717b10d03019c7eb5b6f73ae67edcdd5', 'VALIDATION.txt': '5186e85afdaaab458d699f77e5e53a2b7fc67ff174ef1281cba7a2a698fe0338', 'analyze.py': '740be927397d5b3042e1d9c2feda7b940569f3695a59dd9c6d205c0628aa8f92', 'benchmark.py': '46b2e9ed792ae6663a49a1f91545666e5f9315750c7fd9982419bed38de44d07', 'build_full_device_packet.py': '7bbb113cd3222a8fbe0838b813f64b5def36ef16d7afb562e82a94d5f4551095', 'common.py': '8dfe6fff8dfe6c498f5da554b830c2f571b6c284202386174ffbaf4364a8465b', 'environment.py': 'e82caa67f548c4ca271fd17ecc72495f9d366bfe5a34bde78779e357291b7c9e', 'external_frozen.json': '8bed0d3fbe26e3c85358343e4f285ff337ed45ea73e5527aafb8b6e6a25295a2', 'frozen_rules.json': '661af3b209f5a166cef235c31462ce1daa90a6c92123dec6ef1247372ef04c29', 'full_evaluation_seed.json.gz': '6efb4b0a0f6a3e62a64b0a5ba1f57af820373a3eb5bb50226a728123e6027a31', 'inference_worker.py': 'cfac322aeeb98d32e61aee7d197154572c42dbb5c539f4e82615a5607e19b5ab', 'prepare.py': '9db1ca8b8496803c92ff944f410a1db7eca3c35f659d7b5d9ecba1cc8d6d2fb2', 'run.py': '13723ccea40e8a42447403937df84f881a2236d3e0f14aa6bf8b5381d9f4af4a', 'selection_frozen.json': '4e09894f086d5be42082bcd6edd0e312d8ba8034aaec4789ecde3525ec5735c8', 'source_manifest.json': '232d7fc85bea50edf2a88338e50a443fb23146bc6bc8550107a6fbcce768e730'}
raw=base64.b64decode(PAYLOAD);assert hashlib.sha256(raw).hexdigest()==EXPECTED
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert set(z.namelist())==set(FILES)
    for name,value in FILES.items():
        assert Path(name).name==name
        data=z.read(name);assert hashlib.sha256(data).hexdigest()==value
        (BUNDLE/name).write_bytes(data)
del raw,PAYLOAD
print((BUNDLE/'PROTOCOL.txt').read_text())


## Prepare an isolated environment

Uses the earlier Kaggle-compatible fix: the environment is created **without ensurepip**, and the host's pip installs into it. Kernel packages are unchanged. Missing pinned wheels stop the study rather than substituting a version.


In [ ]:
spec=importlib.util.spec_from_file_location('precision_environment',BUNDLE/'environment.py')
envtools=importlib.util.module_from_spec(spec);spec.loader.exec_module(envtools)
PREP=envtools.ensure_environment(TEMP/'preparation_environment')
probe=subprocess.run([str(PREP),'-I','-c','import numpy,tokenizers;assert numpy.__version__=="2.1.3" and tokenizers.__version__=="0.23.3"'],capture_output=True,text=True,env=RUN_ENV)
if probe.returncode:
    log=OUTPUT/'preparation_install.log'
    with log.open('w') as f:
        status=subprocess.run(envtools.pip_command(PREP,'install','--only-binary=:all:','numpy==2.1.3','tokenizers==0.23.3'),stdout=f,stderr=subprocess.STDOUT,env=RUN_ENV)
    if status.returncode:raise RuntimeError('Pinned preparation installation failed. '+log.read_text()[-3000:])
CFG=TEMP/'configuration.json'
if CFG.exists():assert json.loads(CFG.read_text())==CONFIG,'Changed settings: preserve this run and use new OUTPUT and TEMP directories.'
else:CFG.write_text(json.dumps(CONFIG,indent=2))


## Run the frozen intervention study

Default full coverage: four checkpoint pairs,16score jobs on up to2,079messages, then30small timing jobs for the primary pair. Missing extensions reduce coverage. Allow a multi-hour CPU session; duration depends on hardware. Sessions and package environments are checkpointed. Resume only with the same unchanged host/code/input/settings; a new host needs new output and temporary directories.

The source CSVs are downloaded from pinned author repositories. Overlap checks do not certify independence from historical training. Outputs include no raw texts or model weights. The fixed5% guard must remain the primary result even if a sensitivity point looks better.


In [ ]:
try:
    subprocess.run([str(PREP),str(BUNDLE/'run.py'),str(CFG)],check=True,env=RUN_ENV)
except Exception:
    report=OUTPUT.with_name(OUTPUT.name+'_reports.zip')
    if report.exists():
        from IPython.display import display,FileLink
        print('INCOMPLETE diagnostic report; preserve it and return it for debugging:')
        display(FileLink(str(report)))
    raise


## Return the report

Read `completion.json` and `KEY_FINDINGS.txt` first. Check recovery against matched random, added source-negative alerts, actual fallback rate, cross-runtime decisions, timing speedups and co-resident memory. The source-group/author intervals and randomization reference answer different questions; do not interchange them. A positive result must include its alert/memory cost. FP32 agreement is not clinical truth.


In [ ]:
completion=json.loads((OUTPUT/'completion.json').read_text());assert completion['complete']
print(json.dumps(completion,indent=2))
print((OUTPUT/'KEY_FINDINGS.txt').read_text())
if completion['timing_live_vs_saved_disagreements']:
    print('IMPORTANT: live timing decisions differ from saved scoring; investigate before claiming deterministic mitigation.')
REPORT=OUTPUT.with_name(OUTPUT.name+'_reports.zip')
with zipfile.ZipFile(REPORT) as z:assert z.testzip() is None
from IPython.display import display,FileLink
print('RETURN THIS FILE:',REPORT)
display(FileLink(str(REPORT)))
